# Doanh thu và doanh thu theo khu vực
orders.csv + geography.csv + order_items.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# ==========================================
# 1. ĐỌC VÀ GỘP DỮ LIỆU (Giữ nguyên)
# ==========================================
df_orders = pd.read_csv('../dataset/orders.csv')
df_items = pd.read_csv('../dataset/order_items.csv')
df_geo = pd.read_csv('../dataset/geography.csv')

df_merge = pd.merge(df_orders, df_items, on='order_id')
df_final = pd.merge(df_merge, df_geo, on='zip')

# ==========================================
# 2. TIỀN XỬ LÝ CHO XU HƯỚNG VĨ MÔ
# ==========================================
df_final['order_date'] = pd.to_datetime(df_final['order_date'])
df_final['month_dt'] = df_final['order_date'].dt.to_period('M').dt.to_timestamp()
df_final['revenue'] = (df_final['quantity'] * df_final['unit_price']) - df_final['discount_amount']

# Tổng hợp dữ liệu toàn cục theo tháng
macro_trend = df_final.groupby('month_dt').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum')
).reset_index()

# Tính đường trung bình động 3 tháng (3-Month Moving Average) để thấy rõ xu hướng dài hạn
macro_trend['revenue_ma3'] = macro_trend['total_revenue'].rolling(window=3).mean()

# Tổng hợp dữ liệu theo Khu vực (Region)
region_trend = df_final.groupby(['month_dt', 'region'])['revenue'].sum().unstack().fillna(0)

# ==========================================
# 3. TRỰC QUAN HÓA XU HƯỚNG LỚN
# ==========================================
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(2, 1, height_ratios=[1.5, 1]) # Biểu đồ trên to hơn biểu đồ dưới
fig.suptitle('BỨC TRANH VĨ MÔ: DOANH THU & ĐỘNG LỰC TĂNG TRƯỞNG', fontsize=20, fontweight='bold', y=0.96)

# --- BIỂU ĐỒ 1: Sức khỏe tổng thể (Doanh thu, Sản lượng & Xu hướng dài hạn) ---
ax1 = fig.add_subplot(gs[0])

# Cột sản lượng (nền mờ)
ax1_twin = ax1.twinx()
ax1_twin.bar(macro_trend['month_dt'], macro_trend['total_quantity'], color='gray', alpha=0.3, width=20, label='Tổng số lượng bán')
ax1_twin.set_ylabel('Số lượng sản phẩm (Quantity)', color='gray', fontsize=12)

# Đường doanh thu thực tế
ax1.plot(macro_trend['month_dt'], macro_trend['total_revenue'], color='#2980b9', marker='o', linewidth=2, alpha=0.5, label='Doanh thu thực tế hàng tháng')

# Đường xu hướng cốt lõi (Moving Average)
ax1.plot(macro_trend['month_dt'], macro_trend['revenue_ma3'], color='#c0392b', linewidth=4, label='Xu hướng cốt lõi (MA 3 tháng)')

ax1.set_title('Xu hướng Doanh thu tổng và Sản lượng (Loại bỏ nhiễu thời vụ)', fontsize=16, fontweight='bold', pad=15)
ax1.set_ylabel('Doanh thu ($)', color='black', fontsize=12)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.legend(loc='upper left', fontsize=12)
ax1_twin.legend(loc='upper right', fontsize=12)

# --- BIỂU ĐỒ 2: Động lực Vùng miền (Stacked Area) ---
ax2 = fig.add_subplot(gs[1])
# Dùng Stacked Area để thấy cả quy mô tổng lẫn tỷ trọng của từng vùng
ax2.stackplot(
    region_trend.index, 
    [region_trend[col] for col in region_trend.columns], 
    labels=region_trend.columns, 
    alpha=0.8,
    colors=sns.color_palette("Set2", len(region_trend.columns))
)

ax2.set_title('Cấu trúc Doanh thu theo Khu vực (Region Contribution)', fontsize=16, fontweight='bold', pad=15)
ax2.set_ylabel('Doanh thu ($)', fontsize=12)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.legend(loc='upper left', title='Khu vực (Region)')

plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.show()